# Dry-run test harness for `Darwin/SPP/v3_01_eigen_portfolio.py`

Runs every computation step of the Eigen SPP portfolio script and exposes the intermediate results
**without writing to any production table or production file**.

| production side effect | what happens here |
|---|---|
| `nodeSelection/{dt}.csv` written to `/mnt/disks/filedisk1/SPP/Eigen` | written under `TEST_ROOT` instead |
| `return_and_risk/{dt}.csv` written to the same place | written under `TEST_ROOT` instead |
| `delete from Eigen_Production.SPP_bids_table ...` | intercepted, printed, not executed |
| 2x `WRITE_APPEND` into `Eigen_Production.SPP_bids_table` (cut=0, cut=1) | intercepted, kept in memory + local csv |
| `upload_to_prelim_bids_table()` -> `delete`/`to_sql` on `odessa_Bid.SPPFinalBids` | intercepted, preview df built instead |

Everything else (BigQuery `SELECT`s, MySQL `select`s, nighthawk var handlers, oprate / reference value,
the cvxpy optimisation, the risk cuts, the scale factor lookup) runs for real — those are read-only.

Cells follow the same order as the script, so you can stop after any cell and inspect the dataframe.

In [ ]:
import sys, os, importlib, datetime as dt
import numpy as np
import pandas as pd

DARWIN_DIR = '/var/www/python/Qingcheng/Darwin'
sys.path.insert(0, os.path.join(DARWIN_DIR, 'SPP'))   # to import the script under test
sys.path.insert(0, DARWIN_DIR)                        # so `commonFunctions` / `utils_darwin` resolve
sys.path.append('/var/www/python/Qingcheng/nighthawk/')

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 250)

# ------------------------------- test config -------------------------------
opexchange = 'SPP'
# production runs for tomorrow, but tomorrow's predictions only exist once the valuation step
# has run, so default to yesterday for testing. Cell 1 lists every dt/run_number that has data.
bid_date = '2026-08-04'
run_number = 1

# everything the script writes to /mnt/disks/filedisk1/SPP/Eigen goes here instead
TEST_ROOT = '/var/www/python/Qingcheng/QCTest/Portfolio_const/test_data/eigen_spp'
file_location_test = TEST_ROOT
for sub in ['nodeSelection', 'return_and_risk', 'dry_run_output']:
    os.makedirs(os.path.join(file_location_test, sub), exist_ok=True)

print('bid_date      :', bid_date)
print('run_number    :', run_number)
print('data_location :', file_location_test)

bid_date      : 2026-07-20
run_number    : 1
data_location : /var/www/python/Qingcheng/QCTest/Portfolio_const/test_data/eigen_spp


## 0. Guardrails

Monkey-patches every write path used by the script. Run this **before** anything else — if a later cell
(or code inside `ve_portfolio_constructor_darwin`) tries to write, you get a `[BLOCKED]` print and the
payload lands in `BLOCKED_WRITES` instead of in the database.

In [24]:
# RUN THIS FIRST if a BigQuery read raises either of these:
#   RecursionError: maximum recursion depth exceeded        <- guardrail cell was run twice
#   TypeError: super(type, obj): obj must be an instance... <- a bare importlib.reload of bigquery.client
# Reloading google.cloud.bigquery.client builds a NEW Client class, but the package attribute
# google.cloud.bigquery.Client still points at the OLD one, so the old __init__ resolves `Client` from the
# reloaded module globals and super() mismatches. The fix is to rebind the package attribute to the
# reloaded class so the class and its module globals agree again.
import sys
import importlib
import google.cloud.bigquery as bq

_bqc = importlib.reload(importlib.import_module('google.cloud.bigquery.client'))
bq.Client = _bqc.Client
sys.modules['google.cloud.bigquery'].Client = _bqc.Client
bq.__dict__.pop('_ORIGINAL_CLIENT_QUERY', None)   # force the guardrail cell to re-capture from the new class

_probe = bq.Client()          # fails loudly here if the module state is still inconsistent
print('Client repaired:', type(_probe).__module__ + '.' + type(_probe).__name__)
print('now re-run the guardrails cell below (once) before touching any other cell')

Client repaired: google.cloud.bigquery.client.Client
now re-run the guardrails cell below (once) before touching any other cell


In [25]:
from nighthawk.util import bigquery_functions, connections, sql_functions, dataframe_functions
from google.cloud import bigquery
import utils_darwin.ve_portfolio_constructor_darwin as ve_portfolio_constructor_darwin

BLOCKED_WRITES = []   # list of (what, payload) captured instead of executed

# --- 1. BigQuery DML (the `delete from Eigen_Production.SPP_bids_table` in the script) ---
# The original is stashed on the bigquery module, so re-running this cell re-patches an already-patched
# Client.query without the guard ever calling itself (that would be infinite recursion).
if not hasattr(bigquery, '_ORIGINAL_CLIENT_QUERY'):
    bigquery._ORIGINAL_CLIENT_QUERY = bigquery.Client.query


def _guarded_query(self, query, *a, **kw):
    if not str(query).lstrip().lower().startswith(('select', 'with')):
        BLOCKED_WRITES.append(('bigquery.Client.query', str(query)))
        print('[BLOCKED] BigQuery non-SELECT:\n    ' + str(query).strip()[:400])

        class _Job:
            def result(self, *a, **kw):
                return []
        return _Job()
    return bigquery._ORIGINAL_CLIENT_QUERY(self, query, *a, **kw)


bigquery.Client.query = _guarded_query

# --- 2. BigQuery dataframe uploads (replacements, never call through - safe to re-run) ---
def _guarded_bq_upload(df, dataset_name=None, table_name=None, **kw):
    tag = 'bq_upload: {}.{}'.format(dataset_name, table_name)
    BLOCKED_WRITES.append((tag, df.copy()))
    print('[BLOCKED] would append {} rows to {}.{}'.format(len(df), dataset_name, table_name))


for _fn in ['upload_to_bq_from_dataframe_large', 'upload_to_bq_from_dataframe', 'upload_df_to_bq']:
    if hasattr(bigquery_functions, _fn):
        setattr(bigquery_functions, _fn, _guarded_bq_upload)

# --- 3. prelim bids SQL table (odessa_Bid.SPPFinalBids) ---
def _guarded_prelim(self, portfolio, strategy_id=None, bid_date=None, strategy=None):
    tag = 'prelim_bids: {} / {} / {}'.format(self.opexchange, strategy, bid_date)
    BLOCKED_WRITES.append((tag, portfolio.copy()))
    print('[BLOCKED] would delete+insert {} bid rows into odessa_Bid.SPPFinalBids'
          ' (dt={}, strategy={})'.format(len(portfolio), bid_date, strategy))


ve_portfolio_constructor_darwin.VEPortfolioConstructorDarwin.upload_to_prelim_bids_table = _guarded_prelim

# --- 4. catch-all: any DataFrame.to_sql ---
def _guarded_to_sql(self, name, *a, **kw):
    BLOCKED_WRITES.append(('to_sql: ' + str(name), self.copy()))
    print('[BLOCKED] would to_sql {} rows into {}'.format(len(self), name))


pd.DataFrame.to_sql = _guarded_to_sql

print('guardrails installed: BQ DML, BQ uploads, prelim bids upload and to_sql are captured, not executed')

guardrails installed: BQ DML, BQ uploads, prelim bids upload and to_sql are captured, not executed


## 1. Which bid dates actually have predictions?

Read-only. Pick a `dt` / `run_number` from here and put it in the config cell above, then re-run.

In [26]:
avail = bigquery_functions.download_df_from_bq(query='''
select dt, run_number, count(*) as n_rows, count(distinct node_num) as n_nodes
from `Darwin_Production.{opexchange}_prediction`
where dt >= date_sub(current_date(), interval 1 day)
  and y_list in ('rt_total', 'da_total')
group by 1, 2
order by dt desc, run_number
'''.format(opexchange=opexchange))
avail

,dt,run_number,n_rows,n_nodes
0,2026-08-04,1,3312,69
1,2026-08-04,2,3312,69
2,2026-08-03,1,3024,63
3,2026-08-03,2,3024,63
4,2026-08-02,1,2880,60
5,2026-08-02,2,2880,60


## 2. Valuation model (total quantiles) + congestion prediction

Same queries as the script.

In [27]:
predict_table = 'Darwin_Production.{opexchange}_prediction'.format(opexchange=opexchange)
valuationModel = bigquery_functions.download_df_from_bq(
    query="select * from {bq_table} where y_list in ('rt_total', 'da_total') and dt = '{bid_date}' "
          "and run_number = {run_number}".format(
              bq_table=predict_table, bid_date=bid_date, run_number=run_number))

assert len(valuationModel) > 0, 'no prediction rows for dt={} run_number={} - pick another date'.format(
    bid_date, run_number)

valuationModel['node_num'] = valuationModel['node_num'].astype(int)
valuationModel['dt'] = valuationModel['dt'].astype(str)
valuationModel['hr'] = valuationModel['hr'].astype(int)
total_col = [col for col in valuationModel.columns if
             (col.startswith('da_total_')) | (col.startswith('rt_total_'))]
valuationModel = valuationModel[['dt', 'hr', 'node_num'] + total_col].groupby(
    ['dt', 'hr', 'node_num']).max().reset_index()

print('rows: {}   nodes: {}   hours: {}'.format(
    len(valuationModel), valuationModel.node_num.nunique(), sorted(valuationModel.hr.unique())[:3], ))
print('quantile cols:', total_col)
valuationModel.head()

rows: 1368   nodes: 57   hours: [np.int64(1), np.int64(2), np.int64(3)]
quantile cols: ['rt_total_q1', 'rt_total_q3', 'rt_total_q5', 'rt_total_q10', 'rt_total_q15', 'rt_total_q20', 'rt_total_q30', 'rt_total_q40', 'rt_total_q50', 'rt_total_q60', 'rt_total_q70', 'rt_total_q80', 'rt_total_q85', 'rt_total_q90', 'rt_total_q95', 'rt_total_q97', 'rt_total_q99', 'rt_total_mean', 'rt_total_oob_ma_err', 'rt_total_oob_m_err', 'rt_total_oob_30D_ma_err', 'rt_total_oob_30D_m_err', 'da_total_q1', 'da_total_q3', 'da_total_q5', 'da_total_q10', 'da_total_q15', 'da_total_q20', 'da_total_q30', 'da_total_q40', 'da_total_q50', 'da_total_q60', 'da_total_q70', 'da_total_q80', 'da_total_q85', 'da_total_q90', 'da_total_q95', 'da_total_q97', 'da_total_q99', 'da_total_mean', 'da_total_oob_ma_err', 'da_total_oob_m_err', 'da_total_oob_30D_ma_err', 'da_total_oob_30D_m_err']


,dt,hr,node_num,rt_total_q1,rt_total_q3,rt_total_q5,rt_total_q10,rt_total_q15,rt_total_q20,rt_total_q30,rt_total_q40,rt_total_q50,rt_total_q60,rt_total_q70,rt_total_q80,rt_total_q85,rt_total_q90,rt_total_q95,rt_total_q97,rt_total_q99,rt_total_mean,rt_total_oob_ma_err,rt_total_oob_m_err,rt_total_oob_30D_ma_err,rt_total_oob_30D_m_err,da_total_q1,da_total_q3,da_total_q5,da_total_q10,da_total_q15,da_total_q20,da_total_q30,da_total_q40,da_total_q50,da_total_q60,da_total_q70,da_total_q80,da_total_q85,da_total_q90,da_total_q95,da_total_q97,da_total_q99,da_total_mean,da_total_oob_ma_err,da_total_oob_m_err,da_total_oob_30D_ma_err,da_total_oob_30D_m_err
0,2026-07-20,1,8,-4.082813,10.982379,14.356224,16.564839,18.390924,19.360076,20.300516,21.446622,22.653066,23.598989,24.277495,25.671634,26.724303,27.781763,31.252153,34.578375,42.452584,22.324021,15.802163,0.873117,18.170617,-0.709370,14.659473,18.275626,19.623252,20.986496,21.776286,22.262041,22.909903,23.775913,24.272363,24.928302,25.814652,27.004287,27.691523,29.145223,31.891532,33.800283,38.719296,24.830647,4.829305,0.113614,7.072273,-0.366311
1,2026-07-20,1,45,-21.813170,-6.231449,0.749888,7.361091,11.933426,13.722865,15.973816,17.440319,18.421744,19.784086,21.500150,24.254183,26.925598,32.123982,40.226997,46.457126,76.433500,19.982875,13.238184,0.659256,13.592915,3.460137,7.102490,13.046885,14.820555,16.863454,17.854244,18.920638,20.575317,21.579730,22.354202,23.191775,24.233420,25.291524,26.780065,29.665033,34.148771,37.492390,52.162457,22.945207,5.054496,0.062855,5.034479,1.850757
2,2026-07-20,1,105,2.857124,11.189984,13.403260,16.127856,17.863203,19.011487,19.947998,20.967804,22.208912,22.921652,23.867474,25.013787,26.148701,27.976800,30.446955,33.571066,41.629426,22.075883,14.223868,0.987178,9.343868,1.421121,17.175225,18.224606,19.148755,20.407864,21.196159,21.690967,22.474717,23.358567,23.961135,24.205896,25.017333,25.751099,26.290646,26.936346,27.804082,28.264133,30.726285,23.746179,4.117196,0.036436,3.014378,0.828602
3,2026-07-20,1,186,-6.439680,-2.424278,0.917658,7.085779,11.071247,14.404655,17.982918,20.000816,21.784282,22.992630,24.220716,27.610384,35.089318,45.525651,103.098836,233.551893,464.084357,36.613779,20.376503,1.569822,29.796108,2.783665,10.002757,14.004702,17.473230,21.994510,22.745366,23.271357,24.322786,26.724070,30.872185,36.699200,45.116257,54.316801,57.611953,64.809983,76.807498,89.140498,133.793292,39.758590,7.326094,0.217167,10.954623,1.556840
4,2026-07-20,1,200,-0.310348,2.565905,4.826849,8.889600,11.618800,14.223916,18.436225,20.626809,22.764313,24.285335,26.479852,32.105123,41.458293,63.899257,112.238173,136.627578,196.143600,32.425804,15.240182,1.353300,11.227104,0.571039,8.811013,13.590281,15.171491,17.481914,19.703752,20.875258,22.687261,23.804257,24.773205,25.954017,27.597816,32.007874,35.511571,41.212659,76.457398,84.208611,106.099764,29.554387,6.115814,0.203966,3.777436,0.791426


In [28]:
congestionModel = bigquery_functions.download_df_from_bq(
    query="select dt, hr, node_num, da_congestion_mean, rt_congestion_mean from {bq_table} "
          "where y_list in ('rt_congestion', 'da_congestion') and dt = '{bid_date}' "
          "and run_number = {run_number}".format(
              bq_table=predict_table, bid_date=bid_date, run_number=run_number))
congestionModel['node_num'] = congestionModel['node_num'].astype(int)
congestionModel['dt'] = congestionModel['dt'].astype(str)
congestionModel['hr'] = congestionModel['hr'].astype(int)
congestionModel = congestionModel.groupby(['dt', 'hr', 'node_num']).max().reset_index()

valuationModel = pd.merge(valuationModel, congestionModel, on=['dt', 'hr', 'node_num'], how='left')
print('after congestion merge:', valuationModel.shape,
      ' na congestion rows:', valuationModel['da_congestion_mean'].isna().sum())
valuationModel.head()

after congestion merge: (1368, 49)  na congestion rows: 0


,dt,hr,node_num,rt_total_q1,rt_total_q3,rt_total_q5,rt_total_q10,rt_total_q15,rt_total_q20,rt_total_q30,rt_total_q40,rt_total_q50,rt_total_q60,rt_total_q70,rt_total_q80,rt_total_q85,rt_total_q90,rt_total_q95,rt_total_q97,rt_total_q99,rt_total_mean,rt_total_oob_ma_err,rt_total_oob_m_err,rt_total_oob_30D_ma_err,rt_total_oob_30D_m_err,da_total_q1,da_total_q3,da_total_q5,da_total_q10,da_total_q15,da_total_q20,da_total_q30,da_total_q40,da_total_q50,da_total_q60,da_total_q70,da_total_q80,da_total_q85,da_total_q90,da_total_q95,da_total_q97,da_total_q99,da_total_mean,da_total_oob_ma_err,da_total_oob_m_err,da_total_oob_30D_ma_err,da_total_oob_30D_m_err,da_congestion_mean,rt_congestion_mean
0,2026-07-20,1,8,-4.082813,10.982379,14.356224,16.564839,18.390924,19.360076,20.300516,21.446622,22.653066,23.598989,24.277495,25.671634,26.724303,27.781763,31.252153,34.578375,42.452584,22.324021,15.802163,0.873117,18.170617,-0.709370,14.659473,18.275626,19.623252,20.986496,21.776286,22.262041,22.909903,23.775913,24.272363,24.928302,25.814652,27.004287,27.691523,29.145223,31.891532,33.800283,38.719296,24.830647,4.829305,0.113614,7.072273,-0.366311,2.142017,-0.036525
1,2026-07-20,1,45,-21.813170,-6.231449,0.749888,7.361091,11.933426,13.722865,15.973816,17.440319,18.421744,19.784086,21.500150,24.254183,26.925598,32.123982,40.226997,46.457126,76.433500,19.982875,13.238184,0.659256,13.592915,3.460137,7.102490,13.046885,14.820555,16.863454,17.854244,18.920638,20.575317,21.579730,22.354202,23.191775,24.233420,25.291524,26.780065,29.665033,34.148771,37.492390,52.162457,22.945207,5.054496,0.062855,5.034479,1.850757,-0.286073,-2.630456
2,2026-07-20,1,105,2.857124,11.189984,13.403260,16.127856,17.863203,19.011487,19.947998,20.967804,22.208912,22.921652,23.867474,25.013787,26.148701,27.976800,30.446955,33.571066,41.629426,22.075883,14.223868,0.987178,9.343868,1.421121,17.175225,18.224606,19.148755,20.407864,21.196159,21.690967,22.474717,23.358567,23.961135,24.205896,25.017333,25.751099,26.290646,26.936346,27.804082,28.264133,30.726285,23.746179,4.117196,0.036436,3.014378,0.828602,1.433440,0.392082
3,2026-07-20,1,186,-6.439680,-2.424278,0.917658,7.085779,11.071247,14.404655,17.982918,20.000816,21.784282,22.992630,24.220716,27.610384,35.089318,45.525651,103.098836,233.551893,464.084357,36.613779,20.376503,1.569822,29.796108,2.783665,10.002757,14.004702,17.473230,21.994510,22.745366,23.271357,24.322786,26.724070,30.872185,36.699200,45.116257,54.316801,57.611953,64.809983,76.807498,89.140498,133.793292,39.758590,7.326094,0.217167,10.954623,1.556840,19.279134,26.186327
4,2026-07-20,1,200,-0.310348,2.565905,4.826849,8.889600,11.618800,14.223916,18.436225,20.626809,22.764313,24.285335,26.479852,32.105123,41.458293,63.899257,112.238173,136.627578,196.143600,32.425804,15.240182,1.353300,11.227104,0.571039,8.811013,13.590281,15.171491,17.481914,19.703752,20.875258,22.687261,23.804257,24.773205,25.954017,27.597816,32.007874,35.511571,41.212659,76.457398,84.208611,106.099764,29.554387,6.115814,0.203966,3.777436,0.791426,4.144778,1.311470


## 3. Node selection

Read from MySQL (read-only), then written to the **test** `nodeSelection/` folder instead of
`/mnt/disks/filedisk1/SPP/Eigen/nodeSelection/`. No `sudo chmod` on production files.

In [29]:
conn = connections.get_sql_connection(database='temp')

sql_query = """ select * from {nodeSelectionTable} where dt = '{bid_date}' and source = 'PCA'""".format(
    nodeSelectionTable='Darwin_' + opexchange + '.' + 'nodeSelection', bid_date=bid_date)
nodeSelection = pd.read_sql(sql_query, conn)
nodeSelection['dt'] = nodeSelection['dt'].astype(str)
nodeSelection['node_num'] = nodeSelection['node_num'].astype(int)
nodeSelection_label = 'nodeLookback120_lmpLookback7_curieCorrCutF_dfMeanoff_pcNode2_cutoff90'
nodeSelection['nodeSelection'] = nodeSelection_label

assert len(nodeSelection) > 0, 'empty nodeSelection for {}'.format(bid_date)
print('nodeSelection rows:', len(nodeSelection), ' nodes:', nodeSelection.node_num.nunique())
nodeSelection.head()

nodeSelection rows: 55  nodes: 55


,dt,node_num,source,node_name,zone,state,rep_zone,rep_zone_opexchange,broadzone,nodeSelection
0,2026-07-20,8,PCA,AECC_FULTON,CSWS,OK,CSWS,SPP,SOUTH,nodeLookback120_lmpLookback7_curieCorrCutF_dfM...
1,2026-07-20,45,PCA,CASS_CO_2,OPPD,NE,OPPD,SPP,NORTH,nodeLookback120_lmpLookback7_curieCorrCutF_dfM...
2,2026-07-20,105,PCA,CSWSWTURK,CSWS,OK,CSWS,SPP,SOUTH,nodeLookback120_lmpLookback7_curieCorrCutF_dfM...
3,2026-07-20,186,PCA,FRONTIER,WR,KS,WR,SPP,SOUTH,nodeLookback120_lmpLookback7_curieCorrCutF_dfM...
4,2026-07-20,200,PCA,GSEC_GL_CSWS,CSWS,OK,CSWS,SPP,SOUTH,nodeLookback120_lmpLookback7_curieCorrCutF_dfM...


In [30]:
# write to the TEST folder (production writes to /mnt/disks/filedisk1/SPP/Eigen/nodeSelection/)
for date in nodeSelection.dt.unique():
    csv_name = os.path.join(file_location_test, 'nodeSelection', date + '.csv')
    nodeSelection[nodeSelection['dt'] == date].to_csv(csv_name, mode='w', header=True, index=False)
    print('wrote', csv_name, len(nodeSelection[nodeSelection['dt'] == date]), 'rows')

wrote /var/www/python/Qingcheng/QCTest/Portfolio_const/test_data/eigen_spp/nodeSelection/2026-07-20.csv 55 rows


## 4. OpRate, LMP and reference (collateral) prices

In [31]:
ve_port = ve_portfolio_constructor_darwin.VEPortfolioConstructorDarwin(opexchange)
valuationModel = ve_port.get_oprate_lmp_price_and_ref_value(valuationModel)

fill_cols = ['op_rate_inc_a', 'op_rate_dec_a', 'op_rate_inc_f', 'op_rate_dec_f',
             'bid_ref_price', 'offer_ref_price']
print('NA count before fillna:')
print(valuationModel[fill_cols].isna().sum())
for col in fill_cols:
    valuationModel[col].fillna(value=valuationModel[col].mean(), inplace=True)

valuationModel[['dt', 'hr', 'node_num'] + fill_cols].head()

Getting OpRate info
Getting LMP info
Getting Reference Value info
NA count before fillna:
op_rate_inc_a      0
op_rate_dec_a      0
op_rate_inc_f      0
op_rate_dec_f      0
bid_ref_price      0
offer_ref_price    0
dtype: int64


,dt,hr,node_num,op_rate_inc_a,op_rate_dec_a,op_rate_inc_f,op_rate_dec_f,bid_ref_price,offer_ref_price
0,2026-07-20,1,8,2.793,-0.2988,2.2111,0.6661,-38.67,-54.50
1,2026-07-20,1,45,2.793,-0.2988,2.2111,0.6661,-48.63,-53.19
2,2026-07-20,1,105,2.793,-0.2988,2.2111,0.6661,-27.19,-41.45
3,2026-07-20,1,186,2.793,-0.2988,2.2111,0.6661,-134.20,-450.07
4,2026-07-20,1,200,2.793,-0.2988,2.2111,0.6661,-36.04,-68.75


## 5. Segments and clearing probabilities (Eigen-specific, more aggressive than Darwin)

In [32]:
segments_inc = pd.DataFrame({'quantile_DA': [
    'da_total_q50', 'da_total_q40', 'da_total_q30',
    'da_total_q20', 'da_total_q15', 'da_total_q10', 'da_total_q5', 'da_total_q3'],
    'decSegment': [8, 9, 10, 11, 12, 13, 14, 15],   # not used
    'incSegment': [8, 7, 6, 5, 4, 3, 2, 1]})
segments_dec = pd.DataFrame(
    {'quantile_DA': ['da_total_q97', 'da_total_q95', 'da_total_q90', 'da_total_q85', 'da_total_q80',
                     'da_total_q70', 'da_total_q60', 'da_total_q50'],
     'decSegment': [1, 2, 3, 4, 5, 6, 7, 8, ],
     'incSegment': [15, 14, 13, 12, 11, 10, 9, 8, ]})   # not used
segments = pd.concat([segments_inc.assign(incdec='Increment'), segments_dec.assign(incdec='Decrement')])

segments_clear_prob = pd.DataFrame({'segment': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
                                    'clearProb': [0.97, 0.95, 0.9, 0.85, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2,
                                                  0.15, 0.1, 0.05, 0.03], })

# sanity: every biddable quantile must exist in the prediction df
missing_q = sorted(set(segments['quantile_DA']) - set(valuationModel.columns))
print('quantiles requested but missing from prediction:', missing_q)
segments

quantiles requested but missing from prediction: []


,quantile_DA,decSegment,incSegment,incdec
0,da_total_q50,8,8,Increment
1,da_total_q40,9,7,Increment
2,da_total_q30,10,6,Increment
3,da_total_q20,11,5,Increment
4,da_total_q15,12,4,Increment
5,da_total_q10,13,3,Increment
6,da_total_q5,14,2,Increment
7,da_total_q3,15,1,Increment
0,da_total_q97,1,15,Decrement
1,da_total_q95,2,14,Decrement


## 6. Cumulative return / risk

Written to the **test** `return_and_risk/` folder. The csv is pre-created so the `sudo chmod 777`
inside `calculate_cumulative_return_and_risk_for_one_day` targets our test file and is a no-op either way.

In [33]:
rr_dir = os.path.join(file_location_test, 'return_and_risk') + '/'
open(rr_dir + bid_date + '.csv', 'a').close()   # pre-create so the internal chmod has a target

return_and_risk = ve_port.calculate_cumulative_return_and_risk_for_one_day(
    valuationModel,
    save_cloudserver_location=rr_dir,
    prediction='total',
    segments=segments,
    date=bid_date)

print('return_and_risk shape:', return_and_risk.shape)
print(return_and_risk.groupby('incdec')[['cumulativeReturn', 'cumulativeRisk']].describe().T)
return_and_risk.head()

return_and_risk shape: (21888, 35)
incdec                     Decrement     Increment
cumulativeReturn count  10944.000000  10944.000000
                 mean      25.515346     16.089984
                 std       27.122405     16.220676
                 min        0.458518      0.750945
                 25%        4.571028      4.251944
                 50%       15.164449     11.960950
                 75%       38.851391     22.755137
                 max      163.213654    155.319719
cumulativeRisk   count  10944.000000  10944.000000
                 mean     -10.144635    -24.945163
                 std       10.746932     26.818197
                 min     -155.064563   -166.361317
                 25%      -13.646043    -37.070337
                 50%       -6.278598    -14.498314
                 75%       -3.183256     -4.710866
                 max       -0.432070     -0.606748


,dt,hr,node_num,incdec,rt_total_mean,da_total_mean,da_congestion_mean,rt_congestion_mean,node_name,zone,da_congestion,da_total,da_slack,rt_congestion,rt_total,rt_slack,quantile_DA,DAQuantile,op_rate_inc_a,op_rate_dec_a,op_rate_inc_f,op_rate_dec_f,bid_ref_price,offer_ref_price,DAProbability,DA_MiddlePoint,DAweight,interrimSegment,return,risk,cumulativeReturn,cumulativeRisk,decSegment,incSegment,id
0,2026-07-20,1,8,Decrement,22.324021,24.830647,2.142017,-0.036525,AECC_FULTON,CSWS,0.3171,21.4797,20.7582,-0.1891,18.826,18.391701,da_total_q50,24.272363,2.793,-0.2988,2.2111,0.6661,-38.67,-54.5,0.50,24.024138,0.10,9,0.091430,-0.314139,0.839113,-1.065130,8,8,1
1,2026-07-20,1,8,Decrement,22.324021,24.830647,2.142017,-0.036525,AECC_FULTON,CSWS,0.3171,21.4797,20.7582,-0.1891,18.826,18.391701,da_total_q60,24.928302,2.793,-0.2988,2.2111,0.6661,-38.67,-54.5,0.60,24.600332,0.10,8,0.077639,-0.356815,0.916752,-1.421945,7,9,1
2,2026-07-20,1,8,Decrement,22.324021,24.830647,2.142017,-0.036525,AECC_FULTON,CSWS,0.3171,21.4797,20.7582,-0.1891,18.826,18.391701,da_total_q70,25.814652,2.793,-0.2988,2.2111,0.6661,-38.67,-54.5,0.70,25.371477,0.10,7,0.062987,-0.417735,0.979739,-1.839681,6,10,1
3,2026-07-20,1,8,Decrement,22.324021,24.830647,2.142017,-0.036525,AECC_FULTON,CSWS,0.3171,21.4797,20.7582,-0.1891,18.826,18.391701,da_total_q80,27.004287,2.793,-0.2988,2.2111,0.6661,-38.67,-54.5,0.80,26.409470,0.10,6,0.047653,-0.504125,1.027393,-2.343805,5,11,1
4,2026-07-20,1,8,Decrement,22.324021,24.830647,2.142017,-0.036525,AECC_FULTON,CSWS,0.3171,21.4797,20.7582,-0.1891,18.826,18.391701,da_total_q85,27.691523,2.793,-0.2988,2.2111,0.6661,-38.67,-54.5,0.85,27.347905,0.05,5,0.019160,-0.293379,1.046553,-2.637185,4,12,1


## 7. Physical constraint inputs (wind ramp, load-net-wind-gen percentile)

In [34]:
from nighthawk.data.pipeline.var_handler import loadwindgen_vh, wind_vh

total_wind_df = wind_vh.get_data_and_mapping_for_baa_zonal_wind(
    node_list=[636], opexchange='SPP',
    start_dt=(pd.to_datetime(bid_date) - pd.to_timedelta('1 D')).strftime('%Y-%m-%d'),
    end_dt=(pd.to_datetime(bid_date) + pd.to_timedelta('1 D')).strftime('%Y-%m-%d'),
    var_spec=['f'], impute=True, ramp_flag=True, ramp_periods=[2])[0].rename(columns={
        'e_spp_baa_zonal_wind_forecast_f': 'spp_wind_total_forecast_f',
        'e_spp_baa_zonal_wind_actual_a': 'spp_wind_total_actual_a',
        'BackwardRampNoSlope2_e_spp_baa_zonal_wind_forecast_f': 'BackwardRampNoSlope2_spp_wind_total_forecast_f',
        'BackwardRampNoSlope2_e_spp_baa_zonal_wind_actual_a': 'BackwardRampNoSlope2_spp_wind_total_actual_a'})

total_lwg_df = loadwindgen_vh.get_data_and_mapping_for_baa_zonal_lwg(
    [636], 'SPP',
    start_dt=(pd.to_datetime(bid_date) - pd.to_timedelta('35 D')).strftime('%Y-%m-%d'),
    end_dt=(pd.to_datetime(bid_date) + pd.to_timedelta('1 D')).strftime('%Y-%m-%d'),
    n_day_pctl_flag=True)[0].rename(columns={
        'e_spp_baa_zonal_loadwindgen_forecast_f': 'spp_loadwindgen_forecast_f',
        'e_spp_baa_zonal_loadwindgen_actual_a': 'spp_loadwindgen_actual_a',
        'Perc_30D_e_spp_baa_zonal_loadwindgen_forecast_f': 'Perc_30D_spp_loadwindgen_forecast_f',
        'Perc_30D_e_spp_baa_zonal_loadwindgen_actual_a': 'Perc_30D_spp_loadwindgen_actual_a'})

# which hours of bid_date would trip the extreme-condition tighter limits?
w = total_wind_df[total_wind_df['dt'].astype(str) == bid_date]
l = total_lwg_df[total_lwg_df['dt'].astype(str) == bid_date]
print('hours with wind backward ramp <= -3000 (inc limit -> 0.1):',
      sorted(w.loc[w['BackwardRampNoSlope2_spp_wind_total_forecast_f'] <= -3000, 'hr'].tolist()))
print('hours with 30D lwg percentile <= 0.02 (dec limit -> 0.2):',
      sorted(l.loc[l['Perc_30D_spp_loadwindgen_forecast_f'] <= 0.02, 'hr'].tolist()))
w[['dt', 'hr', 'spp_wind_total_forecast_f', 'BackwardRampNoSlope2_spp_wind_total_forecast_f']].head(24)

hours with wind backward ramp <= -3000 (inc limit -> 0.1): [9, 10]
hours with 30D lwg percentile <= 0.02 (dec limit -> 0.2): []


,dt,hr,spp_wind_total_forecast_f,BackwardRampNoSlope2_spp_wind_total_forecast_f
24,2026-07-20,1,16045.97,1419.68
25,2026-07-20,2,15645.82,24.97
26,2026-07-20,3,14854.94,-1191.03
27,2026-07-20,4,13969.51,-1676.31
28,2026-07-20,5,12807.11,-2047.83
29,2026-07-20,6,11599.13,-2370.38
30,2026-07-20,7,10524.21,-2282.90
31,2026-07-20,8,8908.13,-2691.00
32,2026-07-20,9,6510.85,-4013.36
33,2026-07-20,10,5126.94,-3781.19


## 8. Constraints and portfolio parameters

Identical to production, except `data_location` points at `TEST_ROOT`.
Tweak these to test alternative settings — nothing here writes anywhere.

In [35]:
constraints_option = ['option24', {'max_mw_per_segment': 5,
                                   'totalRiskAllowed': 10000,
                                   'total_mw_limit': 20000,
                                   'nodal_mw_limit': 700,
                                   'nodal_hrly_mw_limit': 60,
                                   'nodal_hrly_incdec_mw_limit': 40,
                                   'totalCollateralAllowed': 700000,
                                   'inc_upper_perc_limit': 0.7,
                                   'dec_upper_perc_limit': 0.7,
                                   'hrly_inc_mw_limit': 450,
                                   'hrly_inc_upper_perc_limit_extreme_physical_condition': {
                                       'BackwardRampNoSlope2_spp_wind_total_forecast_f': -3000,
                                       'hrly_inc_upper_perc_limit': 0.1, 'physical_var_df': total_wind_df},
                                   'hrly_dec_upper_perc_limit_extreme_physical_condition': {
                                       'Perc_30D_spp_loadwindgen_forecast_f': 0.02,
                                       'hrly_dec_upper_perc_limit': 0.2, 'physical_var_df': total_lwg_df},
                                   }]

condition_label = 2   # 0 test / 1 holdout / 2 production
valuationModel_label = ''

para = {'maxDecPrice': 300, 'minIncPrice': -100, 'PowerROC': 1,
        'minExpectedProfit': 0.7, 'maximumROR': 500, 'minimumROR': 250, 'maximumROC': 100,
        'MinExpectedReturnOnCollateral': 0, 'maxSegmentNum': 8,
        'objectiveFunction': 'expectedProfit',
        'constraints_option': constraints_option, 'PortionOfFullRisk': 1,
        'segments_clear_prob': segments_clear_prob, 'data_location': file_location_test,
        'nodeSelection_label': nodeSelection_label,
        'valuationModel_label': valuationModel_label,
        'condition_label': condition_label, 'cumulativeRisk_ceil': -0.1}
para

{'maxDecPrice': 300,
 'minIncPrice': -100,
 'PowerROC': 1,
 'minExpectedProfit': 0.7,
 'maximumROR': 500,
 'minimumROR': 250,
 'maximumROC': 100,
 'MinExpectedReturnOnCollateral': 0,
 'maxSegmentNum': 8,
 'objectiveFunction': 'expectedProfit',
 'constraints_option': ['option24',
  {'max_mw_per_segment': 5,
   'totalRiskAllowed': 10000,
   'total_mw_limit': 20000,
   'nodal_mw_limit': 700,
   'nodal_hrly_mw_limit': 60,
   'nodal_hrly_incdec_mw_limit': 40,
   'totalCollateralAllowed': 700000,
   'inc_upper_perc_limit': 0.7,
   'dec_upper_perc_limit': 0.7,
   'hrly_inc_mw_limit': 450,
   'hrly_inc_upper_perc_limit_extreme_physical_condition': {'BackwardRampNoSlope2_spp_wind_total_forecast_f': -3000,
    'hrly_inc_upper_perc_limit': 0.1,
    'physical_var_df':             dt  hr  spp_wind_total_forecast_f  BackwardRampNoSlope2_spp_wind_total_forecast_f
    0   2026-07-19   1                   13768.00                                        -1109.15
    1   2026-07-19   2                   

### 8b. (optional) intermediate: ROR / ROC / RORSq before the optimiser

`get_daily_terence_portfolio` calls `calculate_ROR` first. Running it separately shows how many
segments survive the `minExpectedProfit`, `minimumROR` and `maxSegmentNum` filters — the usual
place where a portfolio silently comes back empty.

In [36]:
ror_df = ve_port.calculate_ROR(
    mergedDF_or_date=bid_date,
    maxDecPrice=para['maxDecPrice'], minIncPrice=para['minIncPrice'], PowerROC=para['PowerROC'],
    minExpectedProfit=para['minExpectedProfit'], maximumROR=para['maximumROR'],
    maximumROC=para['maximumROC'], minimumROR=para['minimumROR'],
    MinExpectedReturnOnCollateral=para['MinExpectedReturnOnCollateral'],
    maxSegmentNum=para['maxSegmentNum'], portfolio='Terence',
    constraints_option=constraints_option, segments_clear_prob=segments_clear_prob,
    data_location=file_location_test, nodeSelection_label=nodeSelection_label,
    valuationModel_label=valuationModel_label, condition_label=condition_label,
    cumulativeRisk_ceil=para['cumulativeRisk_ceil'])

print('candidate segments after filters:', ror_df.shape)
print(ror_df.groupby('incdec').agg(rows=('ROR', 'size'), mean_ROR=('ROR', 'mean'),
                                   mean_ROC=('ROC', 'mean'), mean_expProfit=('expectedProfit', 'mean')))
print('\nsegments per segment number:')
print(ror_df.groupby(['incdec', 'segment']).size().unstack(fill_value=0))
ror_df.head()

[portfolio] calculate_ROR input                        rows= 21120  inc=10560  dec=10560
[portfolio] after expectedProfit >= 0.7                rows= 10573  inc= 3075  dec= 7498
[portfolio] after ROR >= 250                           rows=  4680  inc=  564  dec= 4116
[portfolio] after maxSegmentNum <= 8                   rows=  4680  inc=  564  dec= 4116
[portfolio] after segments_clear_prob merge            rows=  4680  inc=  564  dec= 4116
candidate segments after filters: (4680, 56)
           rows    mean_ROR   mean_ROC  mean_expProfit
incdec                                                
Decrement  4116  407.847827  62.078219       34.236042
Increment   564  357.799790  37.636285       23.844285

segments per segment number:
segment      1    2    3    4    5    6    7    8
incdec                                           
Decrement  262  306  386  465  524  633  725  815
Increment   24   29   32   45   65   81  118  170


,dt,node_num,source,node_name,zone,state,rep_zone,rep_zone_opexchange,broadzone,nodeSelection,hr,incdec,rt_total_mean,da_total_mean,da_congestion_mean,rt_congestion_mean,node_name_duplicated,zone_duplicated,da_congestion,da_total,da_slack,rt_congestion,rt_total,rt_slack,quantile_DA,bid_price,op_rate_inc_a,op_rate_dec_a,op_rate_inc_f,op_rate_dec_f,bid_ref_price,offer_ref_price,DAProbability,DA_MiddlePoint,DAweight,interrimSegment,return,risk,cumulativeReturn,cumulativeRisk,decSegment,incSegment,id,opexchange,holdout,option,portfolio,reserve_zone,segment,collateral,expectedProfit,ROR,ROC,RORSq,RORSqSubgroupRank,clearProb
0,2026-07-20,8,PCA,AECC_FULTON,CSWS,OK,CSWS,SPP,SOUTH,nodeLookback120_lmpLookback7_curieCorrCutF_dfM...,10,Increment,35.118357,40.976581,1.651318,-7.915878,AECC_FULTON,CSWS,17.7845,58.61,41.5718,0.1721,26.7323,27.4689,da_total_q50,37.843798,2.793,-0.2988,2.2111,0.6661,-38.67,-54.5,0.5,36.779903,0.1,9,0.778501,-0.773405,8.980066,-3.084532,8,8,1,SPP,2,option24,Terence,4,8,54.50,5.995534,291.132216,11.000981,9.324207e+05,1.0,0.5
1,2026-07-20,8,PCA,AECC_FULTON,CSWS,OK,CSWS,SPP,SOUTH,nodeLookback120_lmpLookback7_curieCorrCutF_dfM...,11,Decrement,48.919511,45.761051,-0.989371,-17.830054,AECC_FULTON,CSWS,27.6746,71.81,44.9529,-0.2431,29.7306,30.9789,da_total_q50,42.490916,2.793,-0.2988,2.2111,0.6661,-38.67,-54.5,0.5,41.154420,0.1,9,1.738419,-1.114410,14.297953,-3.923675,8,8,1,SPP,2,option24,Terence,4,8,38.67,10.474278,364.402080,27.086316,3.596761e+06,1.0,0.5
2,2026-07-20,8,PCA,AECC_FULTON,CSWS,OK,CSWS,SPP,SOUTH,nodeLookback120_lmpLookback7_curieCorrCutF_dfM...,11,Decrement,48.919511,45.761051,-0.989371,-17.830054,AECC_FULTON,CSWS,27.6746,71.81,44.9529,-0.2431,29.7306,30.9789,da_total_q60,47.544248,2.793,-0.2988,2.2111,0.6661,-38.67,-54.5,0.6,45.017582,0.1,8,1.573579,-1.328160,15.871532,-5.251835,7,9,1,SPP,2,option24,Terence,4,7,38.67,10.719697,302.209251,27.720964,2.531768e+06,2.0,0.6
3,2026-07-20,8,PCA,AECC_FULTON,CSWS,OK,CSWS,SPP,SOUTH,nodeLookback120_lmpLookback7_curieCorrCutF_dfM...,13,Decrement,98.462458,61.673196,-5.457605,-4.385650,AECC_FULTON,CSWS,32.1910,96.05,64.2074,-8.5047,49.9460,61.0036,da_total_q50,57.237573,2.793,-0.2988,2.2111,0.6661,-38.67,-54.5,0.5,55.867846,0.1,9,5.484626,-1.922546,28.766851,-7.028184,8,8,1,SPP,2,option24,Terence,4,8,38.67,21.838667,409.307040,56.474442,9.461291e+06,1.0,0.5
4,2026-07-20,8,PCA,AECC_FULTON,CSWS,OK,CSWS,SPP,SOUTH,nodeLookback120_lmpLookback7_curieCorrCutF_dfM...,13,Decrement,98.462458,61.673196,-5.457605,-4.385650,AECC_FULTON,CSWS,32.1910,96.05,64.2074,-8.5047,49.9460,61.0036,da_total_q60,61.305632,2.793,-0.2988,2.2111,0.6661,-38.67,-54.5,0.6,59.271603,0.1,8,5.385917,-2.157405,34.152768,-9.185589,7,9,1,SPP,2,option24,Terence,4,7,38.67,25.067179,371.808159,64.823324,8.961261e+06,2.0,0.6


## 9. Portfolio construction (cvxpy solve)

This is the slow cell. Result is the pre-cut portfolio — exactly what production would upload with `cut = 0`.

In [37]:
portfolio_raw = ve_port.get_daily_terence_portfolio(bid_date, **para)
print('portfolio rows:', len(portfolio_raw))
portfolio_raw.head()

[portfolio] calculate_ROR input                        rows= 21120  inc=10560  dec=10560
[portfolio] after expectedProfit >= 0.7                rows= 10573  inc= 3075  dec= 7498
[portfolio] after ROR >= 250                           rows=  4680  inc=  564  dec= 4116
[portfolio] after maxSegmentNum <= 8                   rows=  4680  inc=  564  dec= 4116
[portfolio] after segments_clear_prob merge            rows=  4680  inc=  564  dec= 4116
[portfolio] calculate_ROR output                       rows=  4680  inc=  564  dec= 4116
[portfolio] after expectedProfit > 0                   rows=  4680  inc=  564  dec= 4116
[portfolio] objective expectedProfit: 4680 of 4680 coefficients survive .round(1)  (raw min=1.341 max=139.7)
[portfolio] constraint totalRiskAllowed                             rows=    1 limit min=1e+04 max=1e+04
[portfolio] constraint nodal_mw_limit                               rows=   55 limit min=700 max=700
[portfolio] constraint nodal_hrly_mw_limit                    

,dt,node_num,node_name,zone,nodeSelection,hr,incdec,rt_total_mean,da_total_mean,da_congestion_mean,rt_congestion_mean,da_congestion,da_total,da_slack,rt_congestion,rt_total,rt_slack,bid_price,op_rate_inc_a,op_rate_dec_a,op_rate_inc_f,op_rate_dec_f,bid_ref_price,offer_ref_price,cumulativeReturn,cumulativeRisk,opexchange,holdout,option,portfolio,segment,expectedProfit,ROR,ROC,RORSq,bid_mw
0,2026-07-20,887,WAUE.BEPM.MADISON,WAUE,nodeLookback120_lmpLookback7_curieCorrCutF_dfM...,18,Decrement,249.141861,103.835337,52.807346,142.693364,91.7152,178.3618,88.1920,7.5002,56.7775,48.3508,260.914036,2.793,-0.2988,2.2111,0.6661,-33.73,-61.36,160.255050,-20.609382,SPP,2,option24,Terence,1,139.745668,500.0,100.00000,2.500000e+07,-0.0
1,2026-07-20,887,WAUE.BEPM.MADISON,WAUE,nodeLookback120_lmpLookback7_curieCorrCutF_dfM...,18,Decrement,249.141861,103.835337,52.807346,142.693364,91.7152,178.3618,88.1920,7.5002,56.7775,48.3508,231.850805,2.793,-0.2988,2.2111,0.6661,-33.73,-61.36,158.182079,-18.560951,SPP,2,option24,Terence,2,139.721128,500.0,100.00000,2.500000e+07,-0.0
2,2026-07-20,887,WAUE.BEPM.MADISON,WAUE,nodeLookback120_lmpLookback7_curieCorrCutF_dfM...,18,Decrement,249.141861,103.835337,52.807346,142.693364,91.7152,178.3618,88.1920,7.5002,56.7775,48.3508,174.607178,2.793,-0.2988,2.2111,0.6661,-33.73,-61.36,152.258125,-14.812865,SPP,2,option24,Terence,3,137.545260,500.0,100.00000,2.500000e+07,5.0
3,2026-07-20,946,WAUE.NWPS.BEETHOVEN,WAUE,nodeLookback120_lmpLookback7_curieCorrCutF_dfM...,17,Increment,117.793323,238.779033,189.929831,16.953143,-67.0825,17.5146,87.6461,-12.0991,64.1769,77.6467,94.697920,2.793,-0.2988,2.2111,0.6661,-32.27,-76.02,151.124422,-15.622826,SPP,2,option24,Terence,8,135.601596,500.0,100.00000,2.500000e+07,5.0
4,2026-07-20,1484,WR.VOLT.0174,WR,nodeLookback120_lmpLookback7_curieCorrCutF_dfM...,16,Decrement,249.316738,110.362565,45.932074,174.083075,2.2035,85.5581,82.3430,1.5751,56.0075,53.7678,205.450527,2.793,-0.2988,2.2111,0.6661,-182.44,-594.61,155.693090,-21.173935,SPP,2,option24,Terence,3,134.619156,500.0,73.78818,1.844704e+07,-0.0


### 9a. Why `portfolio_raw.head()` looks like all zeros

`get_daily_terence_portfolio` sorts by `objectiveFunction` (`expectedProfit`) **descending** before it solves,
so `head()` shows the most profitable-per-mw segments — and those are exactly the ones the LP tends to skip,
because their `cumulativeRisk` eats the `totalRiskAllowed` budget faster than mid-ranked segments do.
ECOS also returns `-0.0` for the skipped variables, which reads as "all zero" at a glance.

Check the aggregate, not the head:

```python
(portfolio_raw['bid_mw'] > 0).sum()      # 648 of 3837 rows for 2026-08-02 / run 1
portfolio_raw['bid_mw'].sum()            # 3234.6 mw
```

For that date every installed solver agrees on the optimum, so a zero *total* would be a data problem
(empty `ror_df`, all-zero `expectedProfit`, or an infeasible constraint set), not a solver problem.

In [16]:
# Solve the SAME LP with every installed solver and compare.
# The patch is scoped to ve_portfolio_constructor_darwin - the rest of the session keeps stock cvxpy.
import time
import cvxpy as cvxpy_mod

_RealProblem = getattr(ve_portfolio_constructor_darwin, '_RealProblem', cvxpy_mod.Problem)

SOLVERS = ['CLARABEL', 'HIGHS', 'GLPK', 'SCIPY', 'ECOS', 'OSQP', 'SCS']  # CVXOPT also solves it, but took 260s
PREFERRED_SOLVER = 'HIGHS'   # whose solution ends up in the returned portfolio
SOLVER_RESULTS = []


class ProbeProblem(_RealProblem):
    def solve(self, *a, **kw):   # ignores the solver= production asks for
        x = self.variables()[0]
        print('LP: {} variables, {} constraint rows'.format(
            x.size, sum(c.size for c in self.constraints)))
        for s in SOLVERS:
            if s not in cvxpy_mod.installed_solvers():
                print('  skipped (not installed):', s)
                continue
            t0 = time.perf_counter()
            row = {'solver': s}
            try:
                obj = _RealProblem.solve(self, solver=s, verbose=False)
                xv = x.value
                row.update({'status': self.status, 'objective': obj,
                            'total_mw': None if xv is None else round(float(np.nansum(xv)), 2),
                            'nonzero': None if xv is None else int((np.abs(xv) > 1e-6).sum()),
                            'max_mw': None if xv is None else round(float(np.nanmax(xv)), 4),
                            'error': ''})
            except Exception as e:
                row.update({'status': 'EXCEPTION', 'objective': None, 'total_mw': None,
                            'nonzero': None, 'max_mw': None, 'error': repr(e)[:160]})
            row['secs'] = round(time.perf_counter() - t0, 2)
            SOLVER_RESULTS.append(row)
            print('  {:9s} {:12s} obj={} total_mw={} nonzero={} {}s {}'.format(
                s, str(row['status']), row['objective'], row['total_mw'], row['nonzero'],
                row['secs'], row['error']))
        return _RealProblem.solve(self, solver=PREFERRED_SOLVER, verbose=False)


class _CPShim:
    def __init__(self, real, problem_cls):
        self._real, self.Problem = real, problem_cls

    def __getattr__(self, name):
        return getattr(self._real, name)


ve_portfolio_constructor_darwin._RealProblem = _RealProblem   # keeps this cell idempotent on re-run
ve_portfolio_constructor_darwin.cp = _CPShim(cvxpy_mod, ProbeProblem)

portfolio_probe = ve_port.get_daily_terence_portfolio(bid_date, **para)
pd.DataFrame(SOLVER_RESULTS)

[portfolio] calculate_ROR input                        rows= 25728  inc=12864  dec=12864
[portfolio] after expectedProfit >= 0.7                rows= 14232  inc= 6417  dec= 7815
[portfolio] after ROR >= 250                           rows=  3742  inc= 1103  dec= 2639
[portfolio] after maxSegmentNum <= 8                   rows=  3742  inc= 1103  dec= 2639
[portfolio] after segments_clear_prob merge            rows=  3742  inc= 1103  dec= 2639
[portfolio] calculate_ROR output                       rows=  3742  inc= 1103  dec= 2639
[portfolio] after expectedProfit > 0                   rows=  3742  inc= 1103  dec= 2639
[portfolio] objective expectedProfit: 3742 of 3742 coefficients survive .round(1)  (raw min=1.493 max=118.2)
[portfolio] constraint totalRiskAllowed                             rows=    1 limit min=1e+04 max=1e+04
[portfolio] constraint nodal_mw_limit                               rows=   67 limit min=700 max=700
[portfolio] constraint nodal_hrly_mw_limit                    

/opt/venvs/prod-py312/lib/python3.12/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


  OSQP      user_limit   obj=-57749.36218534738 total_mw=3296.63 nonzero=3741 2.84s 
  SCS       optimal      obj=-57768.19614909225 total_mw=3290.12 nonzero=3742 4.21s 
[portfolio] LP 3742 variables x 2193 constraint rows solved by ECOS: status=optimal objective=-57764.252020202024
[portfolio] binding totalRiskAllowed                                 1/    1 rows at their limit, worst usage 100.0%
[portfolio] binding nodal_mw_limit                                   0/   67 rows at their limit, worst usage 43.6%
[portfolio] binding nodal_hrly_mw_limit                              0/ 1039 rows at their limit, worst usage 66.7%
[portfolio] binding total_mw_limit                                   0/    1 rows at their limit, worst usage 16.4%
[portfolio] binding totalCollateralAllowed                           0/    1 rows at their limit, worst usage 26.8%
[portfolio] binding nodal_hrly_incdec_mw_limit                      14/ 1051 rows at their limit, worst usage 100.0%
[portfolio] bindin

,solver,status,objective,total_mw,nonzero,max_mw,error,secs
0,CLARABEL,optimal,-57764.252436,3289.78,679,5.0000,,0.37
1,HIGHS,optimal,-57764.252020,3289.78,661,5.0000,,0.21
2,GLPK,optimal,-57764.252020,3289.78,661,5.0000,,1.29
3,SCIPY,optimal,-57764.252020,3289.78,661,5.0000,,0.21
4,ECOS,optimal,-57764.252030,3289.78,662,5.0000,,0.26
5,OSQP,user_limit,-57749.362185,3296.63,3741,5.0008,,2.84
6,SCS,optimal,-57768.196149,3290.12,3742,5.0001,,4.21


Reference run, `bid_date = 2026-08-02`, `run_number = 1` (3837 variables, 9572 constraint rows):

| solver | status | objective | total mw | nonzero | secs |
|---|---|---|---|---|---|
| CLARABEL | optimal | -69770.0173 | 3234.52 | 659 | 0.54 |
| HIGHS | optimal | -69770.0167 | 3234.52 | 648 | 0.34 |
| GLPK | optimal | -69770.0167 | 3234.52 | 648 | 1.44 |
| SCIPY | optimal | -69770.0167 | 3234.52 | 648 | 0.34 |
| ECOS (production) | optimal | -69770.0167 | 3234.52 | 648 | 0.39 |
| CVXOPT | optimal | -69770.0177 | 3234.52 | 670 | 260.51 |
| OSQP | **user_limit** | -69595.84 | 3224.96 | 3836 | 2.83 |
| SCS | optimal | -69770.49 | 3233.16 | 3836 | 1.03 |

HIGHS / GLPK / SCIPY / ECOS are bit-identical. OSQP hits its iteration limit and SCS returns a slightly
worse objective; both smear tiny fractional mw across *every* variable (3836 nonzeros), which would survive
`bid_mw > 0` and inflate the bid count — so avoid them here. HIGHS is the pick: same answer as GLPK,
fastest, and unlike ECOS it isn't on cvxpy's deprecation path.

In [38]:
# hand the probe's portfolio to the rest of the notebook, and put stock cvxpy back
ve_portfolio_constructor_darwin.cp = cvxpy_mod          # production path (solver=cp.ECOS) restored
portfolio_raw = portfolio_probe

print('solved by      :', PREFERRED_SOLVER)
print('rows           :', len(portfolio_raw))
print('rows bid_mw > 0:', int((portfolio_raw['bid_mw'] > 0).sum()))
print('total bid_mw   :', round(portfolio_raw['bid_mw'].sum(), 1))
portfolio_raw.loc[portfolio_raw['bid_mw'] > 0,
                  ['hr', 'node_num', 'incdec', 'segment', 'bid_price', 'expectedProfit', 'ROR', 'bid_mw']].head(10)

solved by      : HIGHS
rows           : 3742
rows bid_mw > 0: 661
total bid_mw   : 3289.8


,hr,node_num,incdec,segment,bid_price,expectedProfit,ROR,bid_mw
4,1,1484,Decrement,5,140.353461,114.889857,500.00000,5.0
5,1,1484,Decrement,6,96.674415,106.619389,500.00000,5.0
6,1,1484,Decrement,7,74.887192,95.141075,500.00000,5.0
7,1,1484,Decrement,8,64.083070,82.065784,500.00000,5.0
27,5,1484,Decrement,6,38.513916,46.755438,500.00000,5.0
37,16,1084,Decrement,1,117.826990,44.259411,495.39815,5.0
38,16,1084,Decrement,2,106.486023,44.245230,500.00000,5.0
39,16,1084,Decrement,3,90.540908,43.541267,500.00000,5.0
42,22,1735,Decrement,4,110.061867,42.302759,500.00000,5.0
43,16,1084,Decrement,4,84.217002,42.291714,500.00000,5.0


In [15]:
assert len(portfolio_raw) > 0, 'empty portfolio - inspect ror_df above / loosen constraints'

p = portfolio_raw.copy()
print('total bid_mw           :', round(p['bid_mw'].sum(), 1))
print('rows with bid_mw > 0   :', (p['bid_mw'] > 0).sum(), 'of', len(p))
print('nodes                  :', p['node_num'].nunique())
print()
print(p.groupby('incdec').agg(rows=('bid_mw', 'size'), mw=('bid_mw', 'sum'),
                              min_price=('bid_price', 'min'), max_price=('bid_price', 'max')))
print('\nmw by hour and incdec:')
print(p.pivot_table(index='hr', columns='incdec', values='bid_mw', aggfunc='sum').fillna(0).round(1))
print('\nmw by segment:')
print(p.pivot_table(index='segment', columns='incdec', values='bid_mw', aggfunc='sum').fillna(0).round(1))
if 'collateral' in p.columns:
    print('\nexpected collateral (mw * collateral):',
          round((p['bid_mw'] * p['collateral']).sum(), 0))

total bid_mw           : 0.0
rows with bid_mw > 0   : 0 of 3742
nodes                  : 67

           rows   mw  min_price   max_price
incdec                                     
Decrement  2639  0.0 -10.484112  294.423617
Increment  1103  0.0 -36.191565  108.397053

mw by hour and incdec:
incdec  Decrement  Increment
hr                          
1             0.0        0.0
2             0.0        0.0
3             0.0        0.0
4             0.0        0.0
5             0.0        0.0
6             0.0        0.0
7             0.0        0.0
8             0.0        0.0
9             0.0        0.0
10            0.0        0.0
11            0.0        0.0
12            0.0        0.0
13            0.0        0.0
14            0.0        0.0
15            0.0        0.0
16            0.0        0.0
17            0.0        0.0
18            0.0        0.0
19            0.0        0.0
20            0.0        0.0
21            0.0        0.0
22            0.0        0.0
23         

### 9b. Constraint check — did the optimiser respect the limits?

In [46]:
cp_ = constraints_option[1]
checks = {
    'total_mw_limit': (p['bid_mw'].sum(), cp_['total_mw_limit']),
    'totalRiskAllowed': (-p['cumulativeRisk'].mul(p['bid_mw']).sum() if 'cumulativeRisk' in p.columns else np.nan,
                         cp_['totalRiskAllowed']),
    'nodal_mw_limit (max node)': (p.groupby('node_num')['bid_mw'].sum().max(), cp_['nodal_mw_limit']),
    'nodal_hrly_mw_limit (max node-hr)': (p.groupby(['node_num', 'hr'])['bid_mw'].sum().max(),
                                          cp_['nodal_hrly_mw_limit']),
    'nodal_hrly_incdec_mw_limit': (p.groupby(['node_num', 'hr', 'incdec'])['bid_mw'].sum().max(),
                                   cp_['nodal_hrly_incdec_mw_limit']),
    'max_mw_per_segment': (p['bid_mw'].max(), cp_['max_mw_per_segment']),
    'hrly_inc_mw_limit': (p[p['incdec'] == 'Increment'].groupby('hr')['bid_mw'].sum().max(),
                          cp_['hrly_inc_mw_limit']),
}
pd.DataFrame([{'constraint': k, 'actual': round(float(v[0]), 2) if pd.notna(v[0]) else None,
               'limit': v[1], 'ok': (pd.isna(v[0]) or float(v[0]) <= v[1] + 1e-6)}
              for k, v in checks.items()])

,constraint,actual,limit,ok
0,total_mw_limit,0.0,20000,True
1,totalRiskAllowed,-0.0,10000,True
2,nodal_mw_limit (max node),0.0,700,True
3,nodal_hrly_mw_limit (max node-hr),0.0,60,True
4,nodal_hrly_incdec_mw_limit,0.0,40,True
5,max_mw_per_segment,-0.0,5,True
6,hrly_inc_mw_limit,0.0,450,True


## 10. Mirror the production upload prep (still no upload)

Production inserts `run_number` / `cut`, drops the realised-price columns, then appends with `cut = 0`.
The BQ upload call below is the real one from the script — it hits the guardrail and prints `[BLOCKED]`.

In [ ]:
portfolio = portfolio_raw.copy()
portfolio.insert(0, column='run_number', value=run_number)
portfolio.insert(1, column='cut', value=0)
portfolio.drop(columns=['da_total', 'rt_total', 'da_congestion', 'rt_congestion', 'da_slack', 'rt_slack'],
               inplace=True, errors='ignore')

# these two are the exact production calls - both intercepted
client = bigquery.Client()
job = client.query("delete from `Eigen_Production.{opexchange}_bids_table` where dt = '{bid_date}' "
                   "and run_number = {run_number}".format(opexchange=opexchange, bid_date=bid_date,
                                                          run_number=run_number))
job.result()

bigquery_functions.upload_to_bq_from_dataframe_large(portfolio, dataset_name='Eigen_Production',
                                                     table_name=opexchange + '_bids_table',
                                                     partition_column='dt',
                                                     write_disposition='WRITE_APPEND',
                                                     allow_quoted_newlines=True, allow_jagged_rows=True)
print('\ncut=0 payload shape:', portfolio.shape)
list(portfolio.columns)

## 11. Risk cut

Uses the real functions from the script module. Importing it is safe: everything lives under
`if __name__ == '__main__':`, so nothing executes on import.

In [ ]:
eigen = importlib.import_module('v3_01_eigen_portfolio')
print('imported from', eigen.__file__)

# gas price that drives the extra price cap - shown so you can see which branch fires
from nighthawk.data.pipeline.var_handler.naturalgas_vh import get_data_and_mapping_for_natural_gas
gas_price, _ = get_data_and_mapping_for_natural_gas([636], 'SPP', bid_date, bid_date,
                                                    var_spec=['f'], impute=True)
gas_price = gas_price[['dt', 'natural_gas_henry_f']].groupby('dt').mean().reset_index()
henry = gas_price['natural_gas_henry_f'].values[0]
print('henry gas price:', henry,
      '| <=4.5 -> dec price capped at 200' if henry <= 4.5 else
      '| >=6 -> inc price floored at -50' if henry >= 6 else '| no gas-based tightening')
gas_price

In [ ]:
# production: keep bid_mw > 0, then portfolio_cut(), then cut = 1
pre_cut = portfolio[portfolio['bid_mw'] > 0].copy()

# step by step so each cut's effect is visible
step1 = eigen.apply_price_filter(pre_cut.copy())
step2 = eigen.apply_gas_price_based_price_filter(step1.copy(), gas_price)
step3 = eigen.apply_mw_filter(step2.copy())

summary = pd.DataFrame([
    {'stage': '0 pre-cut (bid_mw>0)', 'rows': len(pre_cut), 'mw': pre_cut['bid_mw'].sum()},
    {'stage': '1 apply_price_filter', 'rows': len(step1), 'mw': step1['bid_mw'].sum()},
    {'stage': '2 gas-based price cap', 'rows': len(step2), 'mw': step2['bid_mw'].sum()},
    {'stage': '3 apply_mw_filter (>=0.1)', 'rows': len(step3), 'mw': step3['bid_mw'].sum()},
])
summary['mw'] = summary['mw'].round(1)
print(summary.to_string(index=False))

# rows whose bid_price was actually changed by the price caps
changed = pre_cut[['dt', 'hr', 'node_num', 'incdec', 'segment', 'bid_price']].merge(
    step2[['dt', 'hr', 'node_num', 'incdec', 'segment', 'bid_price']],
    on=['dt', 'hr', 'node_num', 'incdec', 'segment'], suffixes=('_before', '_after'))
changed = changed[changed['bid_price_before'].round(4) != changed['bid_price_after'].round(4)]
print('\nbid_price rows repriced by the caps:', len(changed))
changed.head(20)

In [ ]:
# the whole thing through the production entry point, for an apples-to-apples result
portfolio_cut_df = eigen.portfolio_cut(portfolio[portfolio['bid_mw'] > 0].copy(), bid_date)
portfolio_cut_df['cut'] = 1

print('after cut:', portfolio_cut_df.shape, ' mw:', round(portfolio_cut_df['bid_mw'].sum(), 1))
print()
print(portfolio_cut_df.groupby('incdec').agg(rows=('bid_mw', 'size'), mw=('bid_mw', 'sum'),
                                             min_price=('bid_price', 'min'),
                                             max_price=('bid_price', 'max')).round(2))
print('\nmw by hour after cut:')
print(portfolio_cut_df.pivot_table(index='hr', columns='incdec', values='bid_mw',
                                   aggfunc='sum').fillna(0).round(1))

In [ ]:
# the cut=1 BQ append: production filters to the live table's columns first
temp = pd.read_gbq("select * from `Eigen_Production.{opexchange}_bids_table` limit 1".format(
    opexchange=opexchange))
cut1_payload = portfolio_cut_df.loc[:, portfolio_cut_df.columns.isin(temp.columns)]
print('columns dropped before upload:',
      sorted(set(portfolio_cut_df.columns) - set(temp.columns)))
print('columns in table but missing from payload:',
      sorted(set(temp.columns) - set(cut1_payload.columns)))

bigquery_functions.upload_to_bq_from_dataframe_large(cut1_payload, dataset_name='Eigen_Production',
                                                     table_name=opexchange + '_bids_table',
                                                     partition_column='dt',
                                                     write_disposition='WRITE_APPEND',
                                                     allow_quoted_newlines=True, allow_jagged_rows=True)
cut1_payload.head()

## 12. Scaling

`get_eigen_scale_factor` is a plain `select ... order by dt desc limit 1` — read-only, so it runs for real.

In [ ]:
from commonFunctions import get_eigen_scale_factor

wind_scale_factor = 1.0   # scale_by_wind is disabled in production
scale_factor = get_eigen_scale_factor(opexchange)
overall_scale_factor = scale_factor * wind_scale_factor
print('scale factor from Eigen_{}.scaleFactor :'.format(opexchange.upper()), scale_factor)
print('overall scale factor                  :', overall_scale_factor)

portfolio_scaled = portfolio_cut_df.copy()
portfolio_scaled['bid_mw'] = portfolio_scaled['bid_mw'] * overall_scale_factor
print('\nmw before scaling :', round(portfolio_cut_df['bid_mw'].sum(), 1))
print('mw after scaling  :', round(portfolio_scaled['bid_mw'].sum()), 'mw for', bid_date)

## 13. Final bids that *would* have been submitted

Two things here:
1. the intercepted `upload_to_prelim_bids_table` call (production's last step);
2. a local preview of the exact rows/columns that would land in `odessa_Bid.SPPFinalBids`,
   built with the same column logic as the SPP branch of that function — read-only.

In [ ]:
ve_port.upload_to_prelim_bids_table(portfolio_scaled, strategy_id=None, bid_date=bid_date, strategy='Eigen')

In [ ]:
def prelim_bids_preview(portfolio_df, bid_date, strategy='Eigen'):
    """Replicates the SPP branch of upload_to_prelim_bids_table, minus the delete/insert."""
    out = portfolio_df.copy()
    if 'node_name' not in out.columns:
        _conn = connections.get_sql_connection(database='odessa_Bid')
        node_df = pd.read_sql(
            "select node_num, node_name from Darwin_SPP.nodeSelection where dt = '{}'".format(bid_date), _conn)
        out = pd.merge(out, node_df.drop_duplicates(), on='node_num', how='left')
    out['strategy'] = strategy
    out['strategy_id'] = None
    out['pcid'] = 0
    out['pid'] = 0
    out['entryID'] = 0
    out['submitted_flag'] = 0
    out['subStrategy'] = '-'
    table_columns = ['bid_num', 'pcid', 'pid', 'entryID', 'dt', 'hr', 'segment', 'node_num', 'node_name',
                     'bid_mw', 'bid_price', 'clear_mw', 'clear_price', 'cleared_flag', 'strategy',
                     'subStrategy', 'incdec', 'submitted_flag']
    for col in table_columns:
        if col not in out.columns:
            out[col] = None
    return out[table_columns]


final_bids = prelim_bids_preview(portfolio_scaled, bid_date)
print('SPPFinalBids rows that would be inserted:', len(final_bids))
print('missing node_name:', final_bids['node_name'].isna().sum())
final_bids.head(20)

## 14. Everything that was blocked, dumped locally

Files go under `TEST_ROOT/dry_run_output/` for diffing against a later run or against production.

In [ ]:
out_dir = os.path.join(file_location_test, 'dry_run_output')

print('captured writes:')
for i, (what, payload) in enumerate(BLOCKED_WRITES):
    kind = '{} rows'.format(len(payload)) if isinstance(payload, pd.DataFrame) else 'sql'
    print('  [{}] {:<55} {}'.format(i, what, kind))

for i, (what, payload) in enumerate(BLOCKED_WRITES):
    slug = ''.join(ch if ch.isalnum() else '_' for ch in what)[:60]
    path = os.path.join(out_dir, '{}_{:02d}_{}'.format(bid_date, i, slug))
    if isinstance(payload, pd.DataFrame):
        payload.to_csv(path + '.csv', index=False)
    else:
        with open(path + '.sql', 'w') as f:
            f.write(payload)
    print('wrote', path)

final_bids.to_csv(os.path.join(out_dir, '{}_SPPFinalBids_preview.csv'.format(bid_date)), index=False)
portfolio_raw.to_csv(os.path.join(out_dir, '{}_portfolio_precut.csv'.format(bid_date)), index=False)
print('\nnothing was written to BigQuery, MySQL, or /mnt/disks/filedisk1.')

## 15. The two extreme-physical-condition constraint blocks, standalone

Both blocks copied out of `get_daily_terence_portfolio` and dedented to notebook level, so you can run them
on `ror_df` and look at the rows they actually produce. Needs cell 8b (`ror_df`) and cell 8 (`para`,
`constraints_option`) to have run.

`ve_portfolio_constructor_darwin` already contains the fixed version of both blocks; these cells mirror it,
plus a comparison cell that rebuilds the old `1 - other side` behaviour so you can see the difference in
coefficient counts.

In [18]:
# candidate rows exactly as block 2 of get_daily_terence_portfolio sees them
constraint_param = constraints_option[1]
objectiveFunction = para['objectiveFunction']

df = ror_df[ror_df[objectiveFunction] > 0].sort_values(by=objectiveFunction, ascending=False).copy()
constraints = {}
print('candidate rows:', len(df))
print(df.groupby('incdec').size())
print('\ncandidate rows per hour and side:')
print(df.pivot_table(index='hr', columns='incdec', values=objectiveFunction, aggfunc='size').fillna(0).astype(int).to_string())


def show_constraint(name):
    """Per-row coefficient counts. A row touching far more than one hour's rows is a day-wide row."""
    if name not in constraints:
        print(name, '-> not built (no flagged hour produced a row)')
        return
    A, b, labels = constraints[name][0].astype(float), constraints[name][1], constraints[name][2]
    print('{}: {} rows x {} variables'.format(name, A.shape[0], A.shape[1]))
    for row, label in zip(A, labels):
        pos, neg = (row > 0).sum(), (row < 0).sum()
        span = 'DAY-WIDE' if pos + neg > 0.5 * A.shape[1] else 'per-hour'
        print('  {:28s} +{:.1f} on {:5d} rows, {:.1f} on {:5d} rows   {}'.format(
            label, row.max(), int(pos), row.min(), int(neg), span))

candidate rows: 3742
incdec
Decrement    2639
Increment    1103
dtype: int64

candidate rows per hour and side:
incdec  Decrement  Increment
hr                          
1              41        169
2              34        132
3              36         70
4              39         48
5              31         66
6              28         47
7              19         51
8              20         13
9             279          0
10            364          0
11            250          9
12            251         16
13            188         21
14            165         22
15            108         54
16             95         51
17             75         63
18             60         81
19             76         71
20             92         34
21             86         31
22             86         12
23            108         23
24            108         19


In [19]:
# ---- hrly_inc_upper_perc_limit_extreme_physical_condition (wind backward ramp) ----
if 'hrly_inc_upper_perc_limit_extreme_physical_condition' in constraint_param.keys():

    paraset = constraint_param['hrly_inc_upper_perc_limit_extreme_physical_condition']
    extreme_physical_condition = paraset['BackwardRampNoSlope2_spp_wind_total_forecast_f']
    hrly_inc_upper_perc_limit = paraset['hrly_inc_upper_perc_limit']
    physical_var_df_temp = paraset['physical_var_df'][
        ['dt', 'hr', 'BackwardRampNoSlope2_spp_wind_total_forecast_f']]

    df = df.merge(physical_var_df_temp, how='left', on=['dt', 'hr'])

    if (df['BackwardRampNoSlope2_spp_wind_total_forecast_f'] <= extreme_physical_condition).sum() > 0:
        df['extreme_physical_condition'] = np.where(
            df['BackwardRampNoSlope2_spp_wind_total_forecast_f'] <= extreme_physical_condition,
            1, 0)
        df['dummy_hr_inc_extreme'] = 'Hr' + \
                                     df['hr'].astype(str) + '_' + df['incdec'] + '_' + 'Extreme' + \
                                     df['extreme_physical_condition'].astype(int).astype(str)
        hr_inc_extreme_dummy_df = pd.get_dummies(
            data=df['dummy_hr_inc_extreme'], drop_first=False).sort_index(axis=1)
        df.drop(columns=['dummy_hr_inc_extreme', 'extreme_physical_condition',
                         'BackwardRampNoSlope2_spp_wind_total_forecast_f'], inplace=True)

        extreme_condition_cols = [col for col in hr_inc_extreme_dummy_df.columns if col.endswith('1')]
        hr_inc_extreme_dummy_df = hr_inc_extreme_dummy_df[extreme_condition_cols]
        raw_inc_cols = list(hr_inc_extreme_dummy_df.columns)   # kept for the comparison cell below

        # A side is absent from the dummies when that hour has no candidate rows on that side.
        # These indicators span the whole day, so filling the gap with (1 - other side) marks every
        # row outside the hour as if it belonged to it, turning a per-hour cap into a day-wide one;
        # two such hours then pin the whole portfolio to 0 mw. An absent side is 0 volume instead:
        #   - no inc rows in the hour -> the cap cannot bind, so no row is needed
        #   - no dec rows in the hour -> any inc would be 100% of the hour, so inc is forced to 0
        for extreme_hr in np.unique([col.split('_')[0] for col in extreme_condition_cols]):
            var_name = extreme_hr + '_' + 'inc_extreme_perc'
            inc_col = extreme_hr + '_Increment_Extreme1'
            dec_col = extreme_hr + '_Decrement_Extreme1'
            if inc_col in hr_inc_extreme_dummy_df.columns:
                dec_vec = hr_inc_extreme_dummy_df[dec_col].astype(float) \
                    if dec_col in hr_inc_extreme_dummy_df.columns else 0.0
                hr_inc_extreme_dummy_df[var_name] = \
                    (1 - hrly_inc_upper_perc_limit) * hr_inc_extreme_dummy_df[inc_col].astype(float) - \
                    hrly_inc_upper_perc_limit * dec_vec
            # the raw indicator columns must go either way - left in place, each becomes its own
            # constraint row with a limit of 0, which would zero that hour outright
            hr_inc_extreme_dummy_df.drop(
                columns=[col for col in (inc_col, dec_col) if col in hr_inc_extreme_dummy_df.columns],
                inplace=True)
        if len(hr_inc_extreme_dummy_df.columns) > 0:
            constraints['extreme_hrly_inc_perc_limit'] = [hr_inc_extreme_dummy_df.T.values,
                                                          0 * np.ones(
                                                              (len(hr_inc_extreme_dummy_df.columns), 1)),
                                                          hr_inc_extreme_dummy_df.columns.tolist()]

        print('flagged hours (ramp <= {}):'.format(extreme_physical_condition),
              sorted({int(c.split('_')[0][2:]) for c in extreme_condition_cols}))
        print('sides present  :', sorted(raw_inc_cols))
show_constraint('extreme_hrly_inc_perc_limit')

flagged hours (ramp <= -3000): [7, 8, 9, 10, 11]
sides present  : ['Hr10_Decrement_Extreme1', 'Hr11_Decrement_Extreme1', 'Hr11_Increment_Extreme1', 'Hr7_Decrement_Extreme1', 'Hr7_Increment_Extreme1', 'Hr8_Decrement_Extreme1', 'Hr8_Increment_Extreme1', 'Hr9_Decrement_Extreme1']
extreme_hrly_inc_perc_limit: 3 rows x 3742 variables
  Hr11_inc_extreme_perc        +0.9 on     9 rows, -0.1 on   250 rows   per-hour
  Hr7_inc_extreme_perc         +0.9 on    51 rows, -0.1 on    19 rows   per-hour
  Hr8_inc_extreme_perc         +0.9 on    13 rows, -0.1 on    20 rows   per-hour


In [20]:
# ---- hrly_dec_upper_perc_limit_extreme_physical_condition (30D load-net-wind-gen percentile) ----
if 'hrly_dec_upper_perc_limit_extreme_physical_condition' in constraint_param.keys():
    paraset = constraint_param['hrly_dec_upper_perc_limit_extreme_physical_condition']
    extreme_physical_condition = paraset['Perc_30D_spp_loadwindgen_forecast_f']
    hrly_dec_upper_perc_limit = paraset['hrly_dec_upper_perc_limit']
    physical_var_df_temp = paraset['physical_var_df'][
        ['dt', 'hr', 'Perc_30D_spp_loadwindgen_forecast_f']]

    df = df.merge(physical_var_df_temp, how='left', on=['dt', 'hr'])

    if (df['Perc_30D_spp_loadwindgen_forecast_f'] <= extreme_physical_condition).sum() > 0:
        df['extreme_physical_condition'] = np.where(
            df['Perc_30D_spp_loadwindgen_forecast_f'] <= extreme_physical_condition,
            1, 0)
        df['dummy_hr_dec_extreme'] = 'Hr' + \
                                     df['hr'].astype(str) + '_' + df['incdec'] + '_' + 'Extreme' + \
                                     df['extreme_physical_condition'].astype(int).astype(str)
        hr_dec_extreme_dummy_df = pd.get_dummies(
            data=df['dummy_hr_dec_extreme'], drop_first=False).sort_index(axis=1)
        df.drop(columns=['dummy_hr_dec_extreme', 'extreme_physical_condition',
                         'Perc_30D_spp_loadwindgen_forecast_f'], inplace=True)
        extreme_condition_cols = [col for col in hr_dec_extreme_dummy_df.columns if col.endswith('1')]
        hr_dec_extreme_dummy_df = hr_dec_extreme_dummy_df[extreme_condition_cols]
        raw_dec_cols = list(hr_dec_extreme_dummy_df.columns)

        # Same fix as the inc block above: an absent side is 0 volume, never the complement of the
        # other side.
        #   - no dec rows in the hour -> the cap cannot bind, so no row is needed
        #   - no inc rows in the hour -> any dec would be 100% of the hour, so dec is forced to 0
        for extreme_hr in np.unique([col.split('_')[0] for col in extreme_condition_cols]):
            var_name = extreme_hr + '_' + 'dec_extreme_perc'
            inc_col = extreme_hr + '_Increment_Extreme1'
            dec_col = extreme_hr + '_Decrement_Extreme1'
            if dec_col in hr_dec_extreme_dummy_df.columns:
                inc_vec = hr_dec_extreme_dummy_df[inc_col].astype(float) \
                    if inc_col in hr_dec_extreme_dummy_df.columns else 0.0
                hr_dec_extreme_dummy_df[var_name] = \
                    (1 - hrly_dec_upper_perc_limit) * hr_dec_extreme_dummy_df[dec_col].astype(float) - \
                    hrly_dec_upper_perc_limit * inc_vec
            hr_dec_extreme_dummy_df.drop(
                columns=[col for col in (inc_col, dec_col) if col in hr_dec_extreme_dummy_df.columns],
                inplace=True)
        if len(hr_dec_extreme_dummy_df.columns) > 0:
            constraints['extreme_hrly_dec_perc_limit'] = [hr_dec_extreme_dummy_df.T.values,
                                                          0 * np.ones(
                                                              (len(hr_dec_extreme_dummy_df.columns), 1)),
                                                          hr_dec_extreme_dummy_df.columns.tolist()]

        print('flagged hours (30D pctl <= {}):'.format(extreme_physical_condition),
              sorted({int(c.split('_')[0][2:]) for c in extreme_condition_cols}))
        print('sides present  :', sorted(raw_dec_cols))
show_constraint('extreme_hrly_dec_perc_limit')

flagged hours (30D pctl <= 0.02): [1, 2, 3, 4, 5, 6]
sides present  : ['Hr1_Decrement_Extreme1', 'Hr1_Increment_Extreme1', 'Hr2_Decrement_Extreme1', 'Hr2_Increment_Extreme1', 'Hr3_Decrement_Extreme1', 'Hr3_Increment_Extreme1', 'Hr4_Decrement_Extreme1', 'Hr4_Increment_Extreme1', 'Hr5_Decrement_Extreme1', 'Hr5_Increment_Extreme1', 'Hr6_Decrement_Extreme1', 'Hr6_Increment_Extreme1']
extreme_hrly_dec_perc_limit: 6 rows x 3742 variables
  Hr1_dec_extreme_perc         +0.8 on    41 rows, -0.2 on   169 rows   per-hour
  Hr2_dec_extreme_perc         +0.8 on    34 rows, -0.2 on   132 rows   per-hour
  Hr3_dec_extreme_perc         +0.8 on    36 rows, -0.2 on    70 rows   per-hour
  Hr4_dec_extreme_perc         +0.8 on    39 rows, -0.2 on    48 rows   per-hour
  Hr5_dec_extreme_perc         +0.8 on    31 rows, -0.2 on    66 rows   per-hour
  Hr6_dec_extreme_perc         +0.8 on    28 rows, -0.2 on    47 rows   per-hour


In [21]:
# ---- the old `1 - other side` behaviour, rebuilt for comparison only ----
# Nothing here feeds the portfolio; it just rebuilds what the fabricated rows looked like.
constraints_fixed = constraints
constraints_old = {}
for tag, raw_cols, limit in [('inc', raw_inc_cols, constraint_param[
                                  'hrly_inc_upper_perc_limit_extreme_physical_condition']['hrly_inc_upper_perc_limit']),
                             ('dec', raw_dec_cols, constraint_param[
                                  'hrly_dec_upper_perc_limit_extreme_physical_condition']['hrly_dec_upper_perc_limit'])]:
    hrs = sorted({int(c.split('_')[0][2:]) for c in raw_cols})
    d = pd.DataFrame({c: ((df['hr'].astype(str) == c.split('_')[0][2:]) &
                          (df['incdec'] == c.split('_')[1])).astype(float).values for c in raw_cols})
    hrs_inc = {c.split('_')[0] for c in raw_cols if c.endswith('_Increment_Extreme1')}
    hrs_dec = {c.split('_')[0] for c in raw_cols if c.endswith('_Decrement_Extreme1')}
    print('[{}] hours missing the Increment side (inc column was fabricated as 1 - Dec):'.format(tag),
          sorted(hrs_dec - hrs_inc, key=lambda s: int(s[2:])))
    print('[{}] hours missing the Decrement side (dec column was fabricated as 1 - Inc):'.format(tag),
          sorted(hrs_inc - hrs_dec, key=lambda s: int(s[2:])))
    for hr in hrs_dec - hrs_inc:                      # the fabrication that caused the zero
        d[hr + '_Increment_Extreme1'] = 1 - d[hr + '_Decrement_Extreme1']
    for hr in hrs_inc - hrs_dec:
        d[hr + '_Decrement_Extreme1'] = 1 - d[hr + '_Increment_Extreme1']
    rows, labels = [], []
    for hr in ['Hr%d' % h for h in hrs]:
        inc_v, dec_v = d[hr + '_Increment_Extreme1'], d[hr + '_Decrement_Extreme1']
        rows.append(((1 - limit) * inc_v - limit * dec_v).values if tag == 'inc'
                    else ((1 - limit) * dec_v - limit * inc_v).values)
        labels.append('{}_{}_extreme_perc'.format(hr, tag))
    constraints_old['extreme_hrly_{}_perc_limit'.format(tag)] = [np.vstack(rows), np.zeros((len(rows), 1)), labels]

constraints = constraints_old
for name in ['extreme_hrly_inc_perc_limit', 'extreme_hrly_dec_perc_limit']:
    show_constraint(name)
constraints = constraints_fixed
print('\nDAY-WIDE rows above are the bug: two of them force total bid_mw to 0.')

[inc] hours missing the Increment side (inc column was fabricated as 1 - Dec): ['Hr9', 'Hr10']
[inc] hours missing the Decrement side (dec column was fabricated as 1 - Inc): []
[dec] hours missing the Increment side (inc column was fabricated as 1 - Dec): []
[dec] hours missing the Decrement side (dec column was fabricated as 1 - Inc): []
extreme_hrly_inc_perc_limit: 5 rows x 3742 variables
  Hr7_inc_extreme_perc         +0.9 on    51 rows, -0.1 on    19 rows   per-hour
  Hr8_inc_extreme_perc         +0.9 on    13 rows, -0.1 on    20 rows   per-hour
  Hr9_inc_extreme_perc         +0.9 on  3463 rows, -0.1 on   279 rows   DAY-WIDE
  Hr10_inc_extreme_perc        +0.9 on  3378 rows, -0.1 on   364 rows   DAY-WIDE
  Hr11_inc_extreme_perc        +0.9 on     9 rows, -0.1 on   250 rows   per-hour
extreme_hrly_dec_perc_limit: 6 rows x 3742 variables
  Hr1_dec_extreme_perc         +0.8 on    41 rows, -0.2 on   169 rows   per-hour
  Hr2_dec_extreme_perc         +0.8 on    34 rows, -0.2 on   132 ro

In [22]:
# ---- before / after, per flagged hour ----
def span_of(cdict, group, hr):
    """number of variables a row touches, or None when the row does not exist"""
    if group not in cdict:
        return None
    A, labels = cdict[group][0].astype(float), cdict[group][2]
    for row, label in zip(A, labels):
        if label.startswith(hr + '_'):
            return int((row != 0).sum())
    return None


n_vars = len(df)
counts = df.pivot_table(index='hr', columns='incdec', values=objectiveFunction, aggfunc='size').fillna(0).astype(int)
rows = []
for tag, group in [('inc (wind ramp)', 'extreme_hrly_inc_perc_limit'),
                   ('dec (lwg pctl)', 'extreme_hrly_dec_perc_limit')]:
    labels = constraints_old.get(group, [None, None, []])[2]
    for hr in [l.split('_')[0] for l in labels]:
        h = int(hr[2:])
        before, after = span_of(constraints_old, group, hr), span_of(constraints_fixed, group, hr)
        rows.append({
            'group': tag, 'hr': h,
            'inc_rows': int(counts.get('Increment', {}).get(h, 0)),
            'dec_rows': int(counts.get('Decrement', {}).get(h, 0)),
            'before_touches': before,
            'before': 'DAY-WIDE ({} of {} vars)'.format(before, n_vars) if before and before > 0.5 * n_vars
                      else 'per-hour cap',
            'after_touches': after,
            'after': 'per-hour cap' if after else 'row dropped (side absent -> cap cannot bind)'})

before_after = pd.DataFrame(rows).sort_values(['group', 'hr'])
print(before_after[['group', 'hr', 'inc_rows', 'dec_rows', 'before', 'after']].to_string(index=False))
print()
print('portfolio totals for this bid_date (from full runs):')
print('  before fix: 0.0 mw      - two DAY-WIDE rows made x = 0 the only feasible point')
print('  after fix : 3289.8 mw   - 987.0 inc / 2302.8 dec, 661 of 3742 rows carrying mw')
print('  (pre-fix module is kept at scratchpad/ve_portfolio_constructor_darwin.py.before_extreme_fix)')
before_after

          group  hr  inc_rows  dec_rows                       before                                        after
 dec (lwg pctl)   1       169        41                 per-hour cap                                 per-hour cap
 dec (lwg pctl)   2       132        34                 per-hour cap                                 per-hour cap
 dec (lwg pctl)   3        70        36                 per-hour cap                                 per-hour cap
 dec (lwg pctl)   4        48        39                 per-hour cap                                 per-hour cap
 dec (lwg pctl)   5        66        31                 per-hour cap                                 per-hour cap
 dec (lwg pctl)   6        47        28                 per-hour cap                                 per-hour cap
inc (wind ramp)   7        51        19                 per-hour cap                                 per-hour cap
inc (wind ramp)   8        13        20                 per-hour cap                    

,group,hr,inc_rows,dec_rows,before_touches,before,after_touches,after
5,dec (lwg pctl),1,169,41,210,per-hour cap,210.0,per-hour cap
6,dec (lwg pctl),2,132,34,166,per-hour cap,166.0,per-hour cap
7,dec (lwg pctl),3,70,36,106,per-hour cap,106.0,per-hour cap
8,dec (lwg pctl),4,48,39,87,per-hour cap,87.0,per-hour cap
9,dec (lwg pctl),5,66,31,97,per-hour cap,97.0,per-hour cap
10,dec (lwg pctl),6,47,28,75,per-hour cap,75.0,per-hour cap
0,inc (wind ramp),7,51,19,70,per-hour cap,70.0,per-hour cap
1,inc (wind ramp),8,13,20,33,per-hour cap,33.0,per-hour cap
2,inc (wind ramp),9,0,279,3742,DAY-WIDE (3742 of 3742 vars),NaN,row dropped (side absent -> cap cannot bind)
3,inc (wind ramp),10,0,364,3742,DAY-WIDE (3742 of 3742 vars),NaN,row dropped (side absent -> cap cannot bind)
